# HeatGuard AI — Phase 4: Two-Stage Hybrid Engine

**Objective:** Train, calibrate, and verify the production Two-Stage Hybrid Engine combining:
1. **Stage 1: Continuous Temperature Departure Regressor** ($T_{max, t+1} - \text{Threshold}_{t+1}$) + Quantile Uncertainty Bounds ($p_{10}, p_{90}$)
2. **Stage 2: Cost-Sensitive Calibrated LightGBM Classifier** + Isotonic Probability Calibration
3. **Stage 3: Physics-Grounded Severity Mapping** (Normal / Warning / Severe / Extreme)

**Key Scientific Advantage:**
- Direct 4-class classification fails on `Extreme` because there are only 2 extreme cases in the 74-year dataset.
- The Stage 1 Regressor learns from **100% of the 187k continuous records**, naturally predicting when physical departure exceeds $+4.0^\circ\text{C}$.

In [ ]:
import os
import sys
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

# Ensure project root is in path
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from models.hybrid_engine import TwoStageHeatwavePredictor
from models.evaluate import compute_metrics, find_optimal_threshold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print(f"Project Root: {REPO_ROOT}")

## 1. Load Model-Ready Dataset and Extract Splits

In [ ]:
data_path = REPO_ROOT / "data" / "model_ready_dataset.csv"
schema_path = REPO_ROOT / "data" / "feature_schema.json"

df = pd.read_csv(data_path)
with open(schema_path, "r", encoding="utf-8") as f:
    schema = json.load(f)

feature_cols = schema["feature_columns"]
clf_target = "target_heatwave_next_day"
reg_target = "target_departure_next_day"

train_df = df[df["split"] == "train"]
val_df = df[df["split"] == "val"]
test_df = df[df["split"] == "test"]

X_train, y_train_clf, y_train_reg = train_df[feature_cols].copy(), train_df[clf_target].to_numpy().astype(int), train_df[reg_target].to_numpy().astype(float)
X_val, y_val_clf, y_val_reg = val_df[feature_cols].copy(), val_df[clf_target].to_numpy().astype(int), val_df[reg_target].to_numpy().astype(float)
X_test, y_test_clf, y_test_reg = test_df[feature_cols].copy(), test_df[clf_target].to_numpy().astype(int), test_df[reg_target].to_numpy().astype(float)

print(f"Train Split: {X_train.shape[0]:,} rows | {y_train_clf.sum():,} Heatwave days ({y_train_clf.mean()*100:.3f}%)")
print(f"Val Split  : {X_val.shape[0]:,} rows | {y_val_clf.sum():,} Heatwave days ({y_val_clf.mean()*100:.3f}%)")
print(f"Test Split : {X_test.shape[0]:,} rows | {y_test_clf.sum():,} Heatwave days ({y_test_clf.mean()*100:.3f}%)")

## 2. Train Two-Stage Hybrid Predictor

In [ ]:
predictor = TwoStageHeatwavePredictor(
    reg_learning_rate=0.03,
    reg_n_estimators=400,
    reg_num_leaves=31,
    clf_learning_rate=0.03,
    clf_n_estimators=350,
    clf_num_leaves=31,
    scale_pos_weight=15.0,
    calibration_method="isotonic",
    random_state=42
)

t0 = time.time()
predictor.fit(
    X_train=X_train,
    y_train_clf=y_train_clf,
    y_train_reg=y_train_reg,
    X_val=X_val,
    y_val_clf=y_val_clf,
    y_val_reg=y_val_reg
)
print(f"Two-Stage Engine trained and calibrated in {time.time() - t0:.2f} seconds.")

# Tune optimal decision threshold on validation set
val_probs = predictor.predict_proba(X_val)[:, 1]
optimal_tau, val_f1 = find_optimal_threshold(y_val_clf, val_probs)
predictor.optimal_threshold = optimal_tau
print(f"Optimized Validation Threshold: tau* = {optimal_tau:.3f} (Val F1: {val_f1:.4f})")

## 3. Evaluation: Stage 1 Continuous Regressor & Uncertainty Bounds

In [ ]:
for split_name, (X_s, y_s_reg, y_s_clf) in (("TRAIN", (X_train, y_train_reg, y_train_clf)), ("VAL", (X_val, y_val_reg, y_val_clf)), ("TEST", (X_test, y_test_reg, y_test_clf))):
    pred_dep = predictor.predict_departure(X_s)
    rmse = np.sqrt(mean_squared_error(y_s_reg, pred_dep))
    mae = mean_absolute_error(y_s_reg, pred_dep)
    r2 = r2_score(y_s_reg, pred_dep)
    hw_mask = y_s_clf == 1
    tail_mae = mean_absolute_error(y_s_reg[hw_mask], pred_dep[hw_mask]) if hw_mask.sum() > 0 else 0.0
    print(f"[{split_name}] RMSE: {rmse:.4f} °C | MAE: {mae:.4f} °C | R² Score: {r2:.4f} | Heatwave-Day MAE: {tail_mae:.4f} °C")

## 4. Evaluation: Stage 2 Calibrated Classifier (Rare-Event Detection)

In [ ]:
for split_name, (X_s, y_s) in (("TRAIN", (X_train, y_train_clf)), ("VAL", (X_val, y_val_clf)), ("TEST", (X_test, y_test_clf))):
    probs = predictor.predict_proba(X_s)[:, 1]
    m = compute_metrics(y_s, probs, threshold=optimal_tau)
    print(f"[{split_name}] PR-AUC: {m['pr_auc']:.4f} | ROC-AUC: {m['roc_auc']:.4f} | F1: {m['f1_heatwave']:.4f} | Precision: {m['precision']:.4f} | Recall: {m['recall']*100:.2f}% | Brier: {m['brier_score']:.4f}")

## 5. Physical Severity Mapping Verification on Test Set (2021–2024)

In [ ]:
test_preds_df = predictor.predict_full(X_test, threshold=optimal_tau)
test_sev_true = test_df["target_severity_next_day"].to_numpy()
test_sev_pred = test_preds_df["predicted_severity"].to_numpy()

print("Ground Truth vs. Hybrid Predicted Severity (Test Set):")
for sev in ["Normal", "Warning", "Severe", "Extreme"]:
    c_true = (test_sev_true == sev).sum()
    c_pred = (test_sev_pred == sev).sum()
    print(f"  {sev:10s} -> True: {c_true:>5} | Predicted: {c_pred:>5}")

## 6. Simulated Live Inference Demonstration

Simulate live single-day prediction requests as received by the Flask REST backend (`/api/predict`).

In [ ]:
# Sample a summer day from Pune and Ahmedabad
pune_sample = test_df[(test_df["city"] == "Pune") & (test_df["month"] == 5)].iloc[0]
ahmedabad_sample = test_df[(test_df["city"] == "Ahmedabad") & (test_df["month"] == 5)].iloc[0]

print("--- LIVE INFERENCE DEMO (PUNE) ---")
pune_out = predictor.predict_single(pune_sample[feature_cols].to_dict())
for k, v in pune_out.items():
    print(f"  {k:25s}: {v}")

print("\n--- LIVE INFERENCE DEMO (AHMEDABAD) ---")
ahm_out = predictor.predict_single(ahmedabad_sample[feature_cols].to_dict())
for k, v in ahm_out.items():
    print(f"  {k:25s}: {v}")

## 7. Serialize Full Production Model

In [ ]:
save_dir = REPO_ROOT / "models" / "saved"
save_dir.mkdir(parents=True, exist_ok=True)

predictor.save(save_dir / "hybrid_predictor.pkl")
joblib.dump(predictor.regressor, save_dir / "departure_regressor.pkl")
joblib.dump(predictor.calibrated_classifier, save_dir / "lgbm_classifier.pkl")

print(f"Complete production predictor successfully serialized to: {save_dir / 'hybrid_predictor.pkl'}")